In [ ]:
import pandas as pd
import glob

# Path to all CSVs (adjust this to your Kaggle download folder)
path = "./CICIDS2017/"  

# Load all CSV files into a list
all_files = glob.glob(path + "*.csv")

# Combine into one DataFrame
df_list = [pd.read_csv(file) for file in all_files]
df = pd.concat(df_list, ignore_index=True)
df = df.sample(n=20000, random_state=42)

print("Combined shape:", df.shape)
print("Columns:", df.columns[:10], "...")
print("Unique labels:", df[' Label'].unique())


In [ ]:
SEED = 42
NUM_FEATURES = 10

In [ ]:
df.dtypes

In [ ]:
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

# Strip spaces from column names
df.columns = df.columns.str.strip()

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns
label_encoders = {}

le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"\nColumn: {col}")
    print(dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
feature_names = df.columns
X = df.drop('Label', axis=1)   # note: 'Label' (no space after cleaning)
y = df['Label']

# Train/test split
SEED = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Original class distribution (train): {np.bincount(y_train)}")

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
pca = PCA(n_components=NUM_FEATURES, random_state=SEED)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test_scaled)

print("Features Selected: "),
for i, component in enumerate(pca.components_):
    important_idx = np.argmax(np.abs(component))
    print(f"Principal Component {i+1}: most important feature is '{feature_names[important_idx]}'")

In [ ]:
X_train = X_train_pca
X_test = X_test_pca

print(X_train.shape, X_test.shape)

In [ ]:
n_layers = 7
n_classes = len(y.unique())

dev = qml.device("default.qubit", wires=NUM_FEATURES)

@qml.qnode(dev)
def qnn_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(NUM_FEATURES))
    qml.StronglyEntanglingLayers(weights, wires=range(NUM_FEATURES))
    return [qml.expval(qml.PauliZ(i)) for i in range(NUM_FEATURES)]

qnode = qml.QNode(qnn_circuit, dev, interface="torch")

class QNNClassifierMulti(nn.Module):
    def __init__(self, n_layers, n_qubits, n_classes):
        super().__init__()
        weight_shapes = {"weights": (n_layers, n_qubits, 3)}
        self.qlayer = qml.qnn.TorchLayer(qnode, weight_shapes)
        self.fc = nn.Linear(n_qubits, n_classes)  # output = n_classes
    
    def forward(self, x):
        x = self.qlayer(x)
        x = self.fc(x)
        return x  # logits for each class

# class QNNClassifier(nn.Module):
#     def __init__(self, n_layers, n_qubits):
#         super().__init__()
#         weight_shapes = {"weights": (n_layers, n_qubits, 3)}
#         self.qlayer = qml.qnn.TorchLayer(qnode, weight_shapes)
#         self.fc = nn.Linear(n_qubits, 1)
    
#     def forward(self, x):
#         x = self.qlayer(x)
#         x = self.fc(x)
#         return x  # No sigmoid here




In [ ]:
# Prepare data (example subset)
X_train_tensor = torch.tensor(X_train[:3000], dtype=torch.float32)
y_train_tensor = torch.tensor(y_train[:3000].values, dtype=torch.long)  # long for CE loss
X_test_tensor = torch.tensor(X_test[:1000], dtype=torch.float32)
y_test_tensor = torch.tensor(y_test[:1000].values, dtype=torch.long)

# Dataloader
from torch.utils.data import TensorDataset, DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Model, loss, optimizer
model_multi = QNNClassifierMulti(n_layers=n_layers, n_qubits=NUM_FEATURES, n_classes=n_classes)
criterion = nn.CrossEntropyLoss() # removed BCEWithLogitsLoss because it is used for binary calssification
optimizer = optim.Adam(model_multi.parameters(), lr=0.01)

# Training loop
epochs = 100
for epoch in range(epochs):
    model_multi.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model_multi(X_batch)              # shape: (batch, n_classes)
        loss = criterion(outputs, y_batch)          # CE loss expects logits + class indices
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    # if (epoch+1) % 10 == 0:
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")


model_multi.eval()
with torch.no_grad():
    logits = model_multi(X_test_tensor)       # (N, n_classes)
    probs = torch.softmax(logits, dim=1)      # class probabilities
    preds = torch.argmax(probs, dim=1).numpy()

acc = accuracy_score(y_test_tensor, preds)
print(f"Multiclass QNN Accuracy: {acc:.4f}")


In [ ]:
print("Classification Report:
", 
      classification_report(
          y_test_tensor, 
          preds, 
          target_names=[str(c) for c in np.unique(y_test_tensor)]
      ))

In [ ]:
cm = confusion_matrix(y_test_tensor, preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Confusion Matrix - Multiclass QNN")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()